In [1]:
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from scipy.spatial.distance import cdist
import scanpy as sc
import torch



In [2]:
adata = sc.read_h5ad('../example.h5ad')
adata.obs

,in_tissue,array_row,array_col,n_counts,X,Y,tcr
AAACAAGTATCTCCCA-1,1,50,102,5625.0,5000,10200,1
AAACACCAATAACTGC-1,1,59,19,9159.0,5900,1900,1
AAACAGAGCGACTCCT-1,1,14,94,2660.0,1400,9400,0
AAACAGGGTCTATATT-1,1,47,13,9020.0,4700,1300,1
AAACAGTGTTCCTGGG-1,1,73,43,9714.0,7300,4300,0
...,...,...,...,...,...,...,...
TTGTTGTGTGTCAAGA-1,1,31,77,7260.0,3100,7700,0
TTGTTTCACATCCAGG-1,1,58,42,7510.0,5800,4200,1
TTGTTTCATTAGTCTA-1,1,60,30,5742.0,6000,3000,1
TTGTTTCCATACAACT-1,1,45,27,20118.0,4500,2700,0


In [3]:
adata.var.index

Index(['AL627309.1', 'AL669831.5', 'LINC00115', 'FAM41C', 'AL645608.3',
       'AL645608.1', 'SAMD11', 'NOC2L', 'KLHL17', 'PLEKHN1',
       ...
       'AC136616.1', 'BX004987.1', 'AC145212.1', 'MAFIP', 'AC011043.1',
       'AL592183.1', 'AC007325.4', 'AC007325.2', 'AL354822.1', 'AC240274.1'],
      dtype='object', length=19379)

In [4]:
#binding_affinity is a (2640, 19379) datafram, all values are 1, row = adata.obs.index, columns = adata.var.index
binding_affinity = pd.read_csv('../binding_affinity.csv', index_col=0)

In [5]:
binding_affinity.index

Index(['AAACAAGTATCTCCCA-1', 'AAACACCAATAACTGC-1', 'AAACAGAGCGACTCCT-1',
       'AAACAGGGTCTATATT-1', 'AAACAGTGTTCCTGGG-1', 'AAACATTTCCCGGATT-1',
       'AAACCACTACACAGAT-1', 'AAACCCGAACGAAATC-1', 'AAACCGGGTAGGTACC-1',
       'AAACCGTTCGTCCAGG-1',
       ...
       'TTGTGGTAGGAGGGAT-1', 'TTGTGGTATAGGTATG-1', 'TTGTGGTGGTACTAAG-1',
       'TTGTGTATGCCACCAA-1', 'TTGTGTTTCCCGAAAG-1', 'TTGTTGTGTGTCAAGA-1',
       'TTGTTTCACATCCAGG-1', 'TTGTTTCATTAGTCTA-1', 'TTGTTTCCATACAACT-1',
       'TTGTTTGTATTACACG-1'],
      dtype='object', length=2640)

In [67]:
class BagsDataset(Dataset):
    def __init__(self, adata, binding_aff, radius=50, output_csv='bags.csv'):
        self.bags = self.create_bags(adata, binding_aff, radius, output_csv)

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        bag = self.bags[idx]
        distances = torch.tensor(bag['distances'], dtype=torch.float32)
        gene_expression = torch.tensor(bag['gene_expression'], dtype=torch.float32)
        affinity_data = torch.tensor(bag['affinity_data'], dtype=torch.float32)
        label = torch.tensor(bag['label'], dtype=torch.float32)
        return distances, gene_expression, affinity_data, label

    def create_bags(self, adata, binding_aff, radius, output_csv):
        spatial_coords_x = adata.obs['X']
        spatial_coords_y = adata.obs['Y']
        spatial_coords = np.array(list(zip(spatial_coords_x, spatial_coords_y)))
        gene_expression = adata.X
        labels = adata.obs['tcr'].values
        barcodes = adata.obs.index.values
        bags = {}
        dist_matrix = cdist(spatial_coords, spatial_coords, metric='euclidean')
        csv_data = []

        for i in range(len(spatial_coords)):
            in_circle = np.where(dist_matrix[i] <= radius)[0]
            gene_data = gene_expression[in_circle].todense()
            circle_barcodes = barcodes[in_circle]
            affinity_data = np.asmatrix(binding_aff.loc[circle_barcodes].values, dtype=np.float32)
            distances = np.asmatrix(dist_matrix[i][in_circle].reshape(-1, 1), dtype=np.float32)

            if i == 0: 
                print(f"Checking data for bag {i}:")
                print(f"Number of cells in this bag: {len(circle_barcodes)}")
                print(f"Sample of circle_barcodes: {circle_barcodes[:5]}")
                print(f"Shape of affinity_data: {affinity_data.shape}")

            bags[i] = {
                'distances': distances,
                'gene_expression': gene_data,
                'affinity_data': affinity_data,
                'label': labels[i]
            }

            bag_barcodes = barcodes[in_circle]
            for barcode in bag_barcodes:
                csv_data.append([i, barcode, labels[i]])

            print(f"Bag {i} has {gene_data.shape[0]} instances")

        df = pd.DataFrame(csv_data, columns=['bag_id', 'barcode', 'label'])
        df.to_csv(output_csv, index=False)
        return bags

def custom_collate_fn(batch):
    distances, gene_expressions, affinity_data, labels = zip(*batch)
    distances = [torch.tensor(np.array(d), dtype=torch.float32) for d in distances]
    gene_expressions = [torch.tensor(np.array(g), dtype=torch.float32) for g in gene_expressions]
    affinity_data = [torch.tensor(np.array(a), dtype=torch.float32) for a in affinity_data]
    labels = torch.tensor(labels, dtype=torch.float32)
    return distances, gene_expressions, affinity_data, labels


In [44]:
def custom_collate_fn(batch):
    distances, gene_expressions, affinity_data, labels = zip(*batch)
    distances = [torch.tensor(np.array(d), dtype=torch.float32) for d in distances]
    gene_expressions = [torch.tensor(np.array(g), dtype=torch.float32) for g in gene_expressions]
    affinity_data = [torch.tensor(np.array(a), dtype=torch.float32) for a in affinity_data]
    labels = torch.tensor(labels, dtype=torch.float32)
    return distances, gene_expressions, affinity_data, labels

In [68]:
dataset = BagsDataset(adata, binding_affinity, radius= 200)
dataloader = DataLoader(dataset, batch_size=1, collate_fn=custom_collate_fn)



Checking data for bag 0:
Number of cells in this bag: 9
Sample of circle_barcodes: ['AAACAAGTATCTCCCA-1' 'ACTAGTTGCGATCGTC-1' 'AGAGGCTTCGGAAACC-1'
 'AGTTTGGCACGGGTTG-1' 'CAGCAGTCCAGACTAT-1']
Shape of affinity_data: (9, 19379)
Bag 0 has 9 instances
Bag 1 has 9 instances
Bag 2 has 9 instances
Bag 3 has 7 instances
Bag 4 has 5 instances
Bag 5 has 9 instances
Bag 6 has 5 instances
Bag 7 has 9 instances
Bag 8 has 8 instances
Bag 9 has 9 instances
Bag 10 has 9 instances
Bag 11 has 9 instances
Bag 12 has 4 instances
Bag 13 has 9 instances
Bag 14 has 9 instances
Bag 15 has 8 instances
Bag 16 has 8 instances
Bag 17 has 9 instances
Bag 18 has 9 instances
Bag 19 has 9 instances
Bag 20 has 9 instances
Bag 21 has 8 instances
Bag 22 has 8 instances
Bag 23 has 9 instances
Bag 24 has 9 instances
Bag 25 has 9 instances
Bag 26 has 9 instances
Bag 27 has 9 instances
Bag 28 has 9 instances
Bag 29 has 9 instances
Bag 30 has 9 instances
Bag 31 has 9 instances
Bag 32 has 9 instances
Bag 33 has 9 instances
Ba

In [69]:
len(dataloader)

2640

In [70]:
first_bag = dataset.bags[0]

In [71]:
first_bag['gene_expression'].shape

(9, 19379)

In [72]:
first_bag['affinity_data'].shape

(9, 19379)

In [73]:
dataloader.dataset.bags[2]['gene_expression']

matrix([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [74]:
dataloader.dataset.bags[2]['affinity_data']

matrix([[1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        ...,
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.]], dtype=float32)

In [75]:
dataloader.dataset.bags[2]['distances']

matrix([[  0.     ],
        [141.42136],
        [200.     ],
        [141.42136],
        [141.42136],
        [141.42136],
        [200.     ],
        [200.     ],
        [200.     ]], dtype=float32)

In [78]:
dataloader.dataset.bags[0]['label']

np.int64(1)

In [63]:
dataset.__getitem__(2)

(tensor([[  0.0000],
         [141.4214],
         [200.0000],
         [141.4214],
         [141.4214],
         [141.4214],
         [200.0000],
         [200.0000],
         [200.0000]]),
 tensor([[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]),
 tensor([[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]]),
 tensor(0.))

In [79]:
for i, (distances, gene_expressions, affinity_data, label) in enumerate(dataloader):
    print(f"Batch {i}")
    print(f"distances: {distances}")
    print(f"gene_expressions: {gene_expressions}")
    print(f"affinity_data: {affinity_data}")
    print(f"label: {label}")

Batch 0
distances: [tensor([[  0.0000],
        [200.0000],
        [141.4214],
        [200.0000],
        [200.0000],
        [200.0000],
        [141.4214],
        [141.4214],
        [141.4214]])]
gene_expressions: [tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])]
affinity_data: [tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        ...,
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.]])]
label: tensor([1.])
Batch 1
distances: [tensor([[  0.0000],
        [200.0000],
        [200.0000],
        [141.4214],
        [141.4214],
        [200.0000],
        [200.0000],
        [141.4214],
        [141.4214]])]
gene_expressions: [tensor([[0.,

process the visium H5AD

In [3]:
adata = sc.read_h5ad('/project/DPDS/Wang_lab/s439765/spatial_tcr/test/HumanColorectalCancer/HumanColorectalCancer.h5ad')

In [7]:
adata.obs['t_gene_signature'].mean()

np.float32(-4.1684413e-07)